# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm

from shared.iterative_regression.OLS.numpy import gaussian_regression as greg

import warnings;warnings.filterwarnings('ignore')

In [2]:
# DATA_PATH = '../../../updated-data/obs/lme-ready'
# NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/obs'
NULL_DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/null-files/null'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [5]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if (('xling-' in f) or ('callfriend' in f) or ('callhome' in f))
    and (not f.startswith('.'))
    and f.endswith('.parquet')
]
len(fs)

1289

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'I'

In [8]:
param_list = 'time_delta'

In [9]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

In [10]:
for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(DATA_PATH, f))

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()
    df['turn_delta'] = df['y_turn_id'] - df['x_turn_id']

    df['self'] = (df['x_speaker'] == df['y_speaker'])
    # df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]

    # find and eliminate bad y_turns
    #   The idea is that some speakers only generate one utterance per
    #   conversation, and thus grouping the way we're doing here yields
    #   errors when calc'ing power-law exponents
    #
    #   (1) .drop dupes on y_speaker, y_turn_id
    #   (2) ["y_speaker"].count_values()
    #   (3) k > 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (df['turn_delta'] > 0) & (df['turn_delta'] <= 200)
    ]

    ##### ALIGNING DATA AND CREATING DIFFERENCES FOR LYAPUNOV EXPONENT
    df = df.sort_values(by=['x_turn_id', 'y_speaker', 'y_turn_id'])
    df.index = range(len(df))

    df['turn_merge'] = df.groupby(by=['x_turn_id', 'y_speaker']).cumcount()

    df = pd.merge(
        left=df[[col for col in list(df) if (col != 'y_speaker')]].loc[df['self']], left_on=['x_turn_id', 'turn_merge'],
        right=df[['x_turn_id', 'turn_merge', COEF_VAR, 'y_speaker']].loc[~df['self']], right_on=['x_turn_id', 'turn_merge'],
        how='left'
    )
    df['dI'] = (df[COEF_VAR+'_x'] - df[COEF_VAR+'_y']).__abs__()

    df = pd.merge(
        left=df, left_on=['x_turn_id', 'y_speaker'],
        right=df[['x_turn_id', 'y_speaker', 'y_turn_id', 'dI']].drop_duplicates(subset=['x_turn_id', 'y_speaker']), right_on=['x_turn_id', 'y_speaker'],
        how='left', suffixes=('', '0')
    )

    # df = df.loc[df['y_turn_id'] !=  df['y_turn_id0']]
    # df = df.loc[df['turn_merge'] > 0]
    df['DI'] = np.log(df['dI'] + 1e-5) - np.log(df['dI0'] + 1e-5)


    ##### The show
    groups = df.groupby(by=['x_speaker', 'y_speaker'])

    for labels, dfi in groups:
        if dfi['turn_delta'].min() <= 3:
            try:
                mod = greg()
                mod.fit(
                    # x=dfi[['turn_merge']].values,
                    x=dfi[['turn_delta']].values,
                    y=dfi['DI'].values
                )

                lam, _ = mod.estimates()

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'lang': lang_id(f),
                        'cid': f.replace('.parquet', ''),
                        'x_speaker': labels[0],
                        'y_speaker': labels[1],
                        'self': labels[0] == labels[1],
                        'lambda': lam.item(),
                        'k': len(dfi)
                    }
                ]

            except Exception:
                error_tracker += 1

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1289 [00:00<?, ?it/s]

In [11]:
# error_tracker[0].head()

In [12]:
# error_tracker[1].head()

In [13]:
# df_regression.isna().sum()

In [15]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [14]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [15]:
split_by = ['lang', 'self']

In [16]:
df0 = df_regression[split_by+['lambda']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['lambda']].groupby(by=split_by).agg('std').reset_index()['lambda'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['lambda']].groupby(by=split_by).agg('count').reset_index()['lambda'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [17]:
df0['t'] = df0['lambda'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t',  'k']].values]

In [18]:
df0.head(n=20)

,lang,self,lambda,std,k,se,t,p
0,cro,False,-0.001654,0.023988,1815,0.000563,-2.936687,0.001680
1,deu,False,-0.003757,0.017306,315,0.000975,-3.852852,0.000071
2,eng,False,-0.005027,0.027549,544,0.001181,-4.256023,0.000012
3,fin,False,-0.002129,0.070249,169,0.005404,-0.393926,0.347067
4,fra,False,-0.001389,0.034398,170,0.002638,-0.526487,0.299620
5,ko,False,-0.007608,0.032204,248,0.002045,-3.720325,0.000123
6,spa,False,-0.004317,0.049361,1114,0.001479,-2.918798,0.001792
7,yid,False,0.036866,0.144779,4,0.072389,0.509272,0.322828
8,zho,False,-0.001550,0.031460,697,0.001192,-1.300620,0.096909


In [19]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'language-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [20]:
split_by = ['self']

In [21]:
df0 = df_regression[split_by+['lambda']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['lambda']].groupby(by=split_by).agg('std').reset_index()['lambda'].values
df0['k'] = df_regression[split_by+['lambda']].groupby(by=split_by).agg('count').reset_index()['lambda'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [22]:
df0['t'] = df0['lambda'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t',  'k']].values]

In [23]:
df0.head(n=20)

,self,lambda,std,k,se,t,p
0,False,-0.002983,0.035289,5076,0.000495,-6.023126,9.156360e-10


In [24]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-results.csv'),
    index=False,
    encoding='utf-8'
)